# ColPali with OpenSearch




In [ ]:
!pip install setuptools==58.2.0
!pip install -q transformers>=4.45.0
!pip install colpali_engine>=0.3.1
!pip install datasets
!pip install opensearch-py
!pip install requests_aws4auth
!pip install ranx
!pip install -q -U "colpali-engine[interpretability]>=0.3.2,<0.4.0"

In [ ]:
from colpali_engine.models import ColPali, ColPaliProcessor
from datasets import load_dataset
import torch
from tqdm import tqdm
import json
import random
from ranx import compare, Qrels, Run
import pprint
from io import BytesIO
from pathlib import Path
from typing import Optional, cast
import numpy as np

import matplotlib.pyplot as plt
import requests
import torch
from colpali_engine.interpretability import (
    get_similarity_maps_from_embeddings,
    plot_all_similarity_maps,
    plot_similarity_map,
)
from colpali_engine.models import ColPali, ColPaliProcessor
from colpali_engine.utils.torch_utils import get_torch_device
from PIL import Image

In [ ]:
model_name = (
    "vidore/colpali-v1.3"
)
colpali_model = ColPali.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",  # Use "cuda:0" for GPU, "cpu" for CPU, or "mps" for Apple Silicon
).eval()

colpali_processor = ColPaliProcessor.from_pretrained(
    model_name
)

In [ ]:
dataset_vs = load_dataset("vespa-engine/gpfg-QA", split="train")#vespa-engine/gpfg-QA
len(dataset_vs)

In [ ]:
from tqdm import tqdm
from datasets import load_dataset
import json
import boto3
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from requests.auth import HTTPBasicAuth
from requests_aws4auth import AWS4Auth
batch_size = 10 #based on GPU's
image_seq_length = 1024 #the parameter of ColPali https://huggingface.co/vidore/colpali-v1.3/blob/main/preprocessor_config.json
# max_pool = []
# mean_pool = []

OpenSearchDomainEndpoint = 'search-opensearchservi-shjckef2t7wo-iyv6rajdgxg6jas25aupuxev6i.us-west-2.es.amazonaws.com'
service = 'es'
credentials = boto3.Session().get_credentials()
awsauth = HTTPBasicAuth("<username>,<password>") # put your master credentials here
headers = { "Content-Type": "application/json"}
batch = 0
count = 0
body_ = ''
batch_size = 10
aos_client = OpenSearch(
    hosts = [{'host': OpenSearchDomainEndpoint, 'port': 443}],
    http_auth = awsauth,
    use_ssl = True,
    connection_class = RequestsHttpConnection
)

In [ ]:
from tqdm import tqdm
from datasets import load_dataset
import json
import boto3
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from requests.auth import HTTPBasicAuth
from requests_aws4auth import AWS4Auth
batch_size = 10 #based on GPU's
image_seq_length = 1024 #the parameter of ColPali https://huggingface.co/vidore/colpali-v1.3/blob/main/preprocessor_config.json
# max_pool = []
# mean_pool = []

batch = 0
count = 0
body_ = ''
batch_size = 10
aos_client = OpenSearch(
    hosts = [{'host': OpenSearchDomainEndpoint, 'port': 443}],
    http_auth = awsauth,
    use_ssl = True,
    connection_class = RequestsHttpConnection
)
image_embeddings = []            
def embed_and_upload_batch(image_batch, payload_batch, id_start):
    batch_size_current = len(image_batch)

    with torch.no_grad():
        batch_images = colpali_processor.process_images(image_batch).to(colpali_model.device)
        image_embeddings = colpali_model(**batch_images)
    #return image_embeddings
    # Process max and mean pooled embeddings per row of image (PDF page) grid
    special_tokens = image_embeddings[:, image_seq_length:, :]
    max_pool = torch.cat((torch.max(image_embeddings[:, :image_seq_length, :].reshape((batch_size_current, 32, 32, 128)), dim=2).values, special_tokens), dim=1)
    mean_pool = torch.cat((torch.mean(image_embeddings[:, :image_seq_length, :].reshape((batch_size_current, 32, 32, 128)), dim=2), special_tokens), dim=1)
    print(id_start)
    for k,v in enumerate(payload_batch):
        v['page_sub_vectors'] = []
        for l in mean_pool[k]:
            v['page_sub_vectors'].append({"page_sub_vector":l.tolist()})
        #print(v)
        try:
            
            #print(item)
            print(id_start)
            response_ = aos_client.index(
            index = 'colpali-vs',
            body = v
            )



        #t.sleep(1)

        except Exception as e:
            print(f"Error during upsert: {e}")

with tqdm(total=len(dataset_vs), desc="Uploading Progress") as pbar:
    for i in range(0, len(dataset_vs), batch_size):
        batch = dataset_vs[i : i + batch_size]
        images = batch["full_image"]
        batch_size_current = len(images)
        try:
            rest_payload = []
            for j in range(i, i + batch_size_current):
                itm = dataset_vs[j]
                #print(itm)
                img_format = itm['full_image'].format
                img_path = 'vs/vs_'+str(j)+'.'+img_format
                itm['full_image'].save(img_path)
                rest_payload.append({"image": img_path,
                                    'id1': itm['id'],
                                     'url': itm['url'],
                                     'title': itm['title'],
                                     'page_number': itm['page_number'],
                                     'text':itm['text'],
                                     'queries': itm['queries'],
                                     'questions':itm['questions'] ,
                                    'year': itm['year']}
                                   )
            #print(rest_payload)
            #break
            a = embed_and_upload_batch(
                images,
                rest_payload,
                i

            )
        except Exception as e:
            print(f"Error during upsert: {e}")
            continue
        # Update the progress bar
        #break
        pbar.update(batch_size_current)
print("Uploading complete!")

In [ ]:
vs_queries = dataset_vs['queries']
vs_questions = dataset_vs['questions']
question_ = random.sample(vs_queries, 100)#['financial statements norwegian pension fund']
q_ = []
for q in question_:
    if(len(q)!=0):
        q_.append(q[0])
        
print(q_)
question = q_[:100]    

## Encode Query with Colpali

## what is the proportion of female new hires 2021-2023?

In [ ]:
question = ["proportion of female new hires 2021-2023?"] # query 2: palm oil production risk management

# remove unwanted tokens

In [ ]:
def colpali_query(query): #per query
    with torch.no_grad():
        batch_query = colpali_processor.process_queries([query]).to(
            colpali_model.device
        )
        mask_without_pad = batch_query.input_ids.bool().unsqueeze(-1)
    query_embedding = colpali_model(**batch_query)
    return query_embedding[0].cpu().float().numpy().tolist()
queries_sample_embeddings = [colpali_query(sample) for sample in question]
print("total number of token vectors:"+str(len(queries_sample_embeddings[0])))
queries_sample_embeddings

## Retrieve and re-rank using MaxSim

In [ ]:
from pprint import pprint
batch_size = 8
final_docs_sorted_20 = []
for i in queries_sample_embeddings:
    batch_embeddings = i
    a = np.array(batch_embeddings)
    
    # mean pooling of query token vectors 
    # Alternatively use HybriSearch with upto 5 token queries
    vec = a.mean(axis=0)
    hits = []
    
    query = {
        "size": 200,  # Upsampled by 10x
          "query": {
            "nested": {
              "path": "page_sub_vectors",
              "query": {
                "knn": {
                  "page_sub_vectors.page_sub_vector": {
                    "vector": vec.tolist(),
                    "k": 200
                  }
                }
              }
                }
              }
            }
    response = aos_client.search(
        body = query,
        index = 'colpali-vs'
    )
    
    token_vectors = batch_embeddings
    final_docs = []
    hits += response['hits']['hits']
    
    # MaxSim Re-ranking, get top 20
    
    # torch.einsum is the better performant way to compute MaxSim 
    for ind,j in enumerate(hits):   
        max_score_dict_list = []
        doc={"id":j["_id"],"score":j["_score"],"image":j["_source"]["image"]}
        with_s = j['_source']['page_sub_vectors']
        add_score = 0

        for index,i in enumerate(token_vectors):  # iterate through every token vector
            query_token_vector = np.array(i)
            scores = []
            for m in with_s:             # compare the current token vector with every patch vector
                doc_token_vector = np.array(m['page_sub_vector'])
                score = np.dot(query_token_vector,doc_token_vector)  # perform dot product
                scores.append(score)
                
            scores.sort(reverse=True)
            max_score = scores[0]   # get the max similar patch for the token
            add_score+=max_score    # Sum of all max scores
            
        doc["maxsim_score"] = add_score
        final_docs.append(doc)
    final_docs_sorted = sorted(final_docs, key=lambda d: d['maxsim_score'], reverse=True)
    final_docs_sorted_20.append(final_docs_sorted[:20])
   
print("top 20 pages\n")    
pprint(final_docs_sorted_20[0])


## Display the top result

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    """
    Load a PIL image from a valid URL.
    """
    #response = requests.get(url)
    return Image.open(url)


def scale_image(image: Image.Image, new_height: int = 1024) -> Image.Image:
    """
    Scale an image to a new height while maintaining the aspect ratio.
    """
    # Calculate the scaling factor
    width, height = image.size
    aspect_ratio = width / height
    new_width = int(new_height * aspect_ratio)

    # Resize the image
    scaled_image = image.resize((new_width, new_height))

    return scaled_image
# ==========================     USER INPUTS     ==========================
instance_to_test = 0
top_pdf_ann_rerank = final_docs_sorted_20[0][instance_to_test]['image']
top_pdf_ann_rerank
image_filepath: Optional[str] = None  # "shift_kazakhstan.jpg"
query: str = question[0]

# =========================================================================

if image_filepath:
    assert Path(image_filepath).is_file(), f"Cannot find the image file at `{image_filepath}`"
    image = Image.open(image_filepath)
else:
    image = load_image_from_url(top_pdf_ann_rerank)
# Preview the image
scale_image(image, 1024)

## Generate response using the retrieved page with Amazon Nova VLM

### what is the proportion of female new hires 2021-2023?

In [ ]:
import boto3
from IPython.display import display, Markdown
import base64

client = boto3.client("bedrock-runtime")

def call_nova(
    model,
    messages,
    system_message="",
    streaming=False,
    max_tokens=512,
    temp=0.0001,
    top_p=0.99,
    top_k=20,
    tools=None,
    verbose=False,
):
    client = boto3.client("bedrock-runtime")
    system_list = [{"text": system_message}]
    inf_params = {
        "max_new_tokens": max_tokens,
        "top_p": top_p,
        "top_k": top_k,
        "temperature": temp,
    }
    request_body = {
        "messages": messages,
        "system": system_list,
        "inferenceConfig": inf_params,
    }
    if tools is not None:
        tool_config = []
        for tool in tools:
            tool_config.append({"toolSpec": tool})
        request_body["toolConfig"] = {"tools": tool_config}
    if verbose:
        print("Request Body", request_body)
    if not streaming:
        response = client.invoke_model(modelId=model, body=json.dumps(request_body))
        model_response = json.loads(response["body"].read())
        return model_response, model_response["output"]["message"]["content"][0]["text"]
    else:
        response = client.invoke_model_with_response_stream(
            modelId=model, body=json.dumps(request_body)
        )
        return response["body"]
def get_base64_encoded_value(media_path):
    with open(media_path, "rb") as media_file:
        binary_data = media_file.read()
        base_64_encoded_data = base64.b64encode(binary_data)
        base64_string = base_64_encoded_data.decode("utf-8")
        return base64_string
    

def print_output(content_text):
    display(Markdown(content_text))
system_message = "given an image of a PDF page, answer the question. Be accurate to the question. If you don't find the answer in the page, please say, I don't know"
messages = [
    {
        "role": "user",
        "content": [
            {
                "image": {
                    "format": "jpeg",
                    "source": {
                        "bytes": get_base64_encoded_value(
                            top_pdf_ann_rerank
                        )
                    },
                }
            },
            {
                "text": question[0]#"what is the proportion of female new hires 2021-2023?"
            },
        ],
    }
]
model_response, content_text = call_nova(
    "amazon.nova-pro-v1:0", messages, system_message=system_message, max_tokens=300
)

print("\n[Response Content Text]")
print_output(content_text)

## Similarity Map for interpretability of results

In [ ]:
# Reference from : https://github.com/tonywu71/colpali-cookbooks/blob/main/examples/gen_colpali_similarity_maps.ipynb

# Preprocess inputs
batch_images = colpali_processor.process_images([image]).to(colpali_model.device)

#colpali_processor.process_images([dataset[0]["image"]]).to(colpali_model.device)
batch_queries = colpali_processor.process_queries([question[0]]).to(colpali_model.device)

# Forward passes
with torch.no_grad():
    image_embeddings = colpali_model.forward(**batch_images)
    query_embeddings = colpali_model.forward(**batch_queries)
# Get the number of image patches
n_patches = colpali_processor.get_n_patches(image_size=image.size, patch_size=colpali_model.patch_size)

print(f"Number of image patches: {n_patches}")

# Get the tensor mask to filter out the embeddings that are not related to the image
image_mask = colpali_processor.get_image_mask(batch_images)

# Generate the similarity maps
batched_similarity_maps = get_similarity_maps_from_embeddings(
    image_embeddings=image_embeddings,
    query_embeddings=query_embeddings,
    n_patches=n_patches,
    image_mask=image_mask,
)

# Get the similarity map for our (only) input image
similarity_maps = batched_similarity_maps[0]  # (query_length, n_patches_x, n_patches_y)

print(f"Similarity map shape: (query_length, n_patches_x, n_patches_y) = {tuple(similarity_maps.shape)}")

# Use this cell output to choose a token using its index
query_content = colpali_processor.decode(batch_queries.input_ids[0]).replace(colpali_processor.tokenizer.pad_token, "")
query_content = query_content.replace(colpali_processor.query_augmentation_token, "").strip()
query_tokens = colpali_processor.tokenizer.tokenize(query_content)

pprint({idx: val for idx, val in enumerate(query_tokens)})


In [ ]:
# Choose a token using its index
token = input("Enter the token position")
token_idx = int(token)

print(f"Selected token: `{query_tokens[token_idx]}`")


# Retrieve the similarity map for the chosen token
current_similarity_map = similarity_maps[token_idx]  # (n_patches_x, n_patches_y)

fig, ax = plot_similarity_map(
    image=image,
    similarity_map=current_similarity_map,
    figsize=(8, 8),
    show_colorbar=False,
)

max_sim_score = similarity_maps[token_idx, :, :].max().item()
ax.set_title(f"Token #{token_idx}: `{query_tokens[token_idx]}`. MaxSim score: {max_sim_score:.2f}", fontsize=14)

fig

In [ ]:
answers = []

for i in question:
    print(i)
    response = aos_client.search(
        body = {"query":{"match":{"queries":i}},"size":1},
        index = 'colpali-vs'
    )
    #print(response)
    answers.append(response['hits']['hits'][0]['_source']['image'])
eval_ = []
for id_1,j in enumerate(answers):
    exists = False
    for id_2,k in enumerate(final_docs_sorted_20[id_1]):
        if(j == k['image']):
            exists = True
            eval_.append({"exists":exists,"position":id_2})
            
            break
    if(exists == False):
        eval_.append({"exists":exists,"position":-1})

(eval_)

In [ ]:
plots = plot_all_similarity_maps(
    image=image,
    query_tokens=query_tokens,
    similarity_maps=similarity_maps,
    figsize=(8, 8),
    show_colorbar=False,
    add_title=True,
)

for idx, (fig, ax) in enumerate(plots):
    savepath = f"similarity_map_{idx}.png"
    fig.savefig(savepath, bbox_inches="tight")
    print(f"Similarity map for token `{query_tokens[idx]}` saved at `{savepath}`")

plt.close("all")